# 🤖 Notebook 3: Model Building & Evaluation
**HR Employee Attrition Project**

**Objective:** Train classification models to predict which employees are likely to leave. Evaluate models rigorously using multiple metrics beyond just accuracy.

---

## 1. Setup & Load Cleaned Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/cleaned_hr_data.csv')
print(f'Cleaned dataset loaded: {df.shape}')
df.head(2)

## 2. Prepare Features & Target

In [ ]:
X = df.drop('Attrition', axis=1)
y = df['Attrition']

print(f'Features: {X.shape[1]}')
print(f'Target distribution:')
print(y.value_counts())
print(f'\nAttrition rate: {y.mean()*100:.1f}%')

## 3. Train/Test Split

In [ ]:
# Stratified split ensures both sets maintain the same attrition ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} rows')
print(f'Test set: {X_test.shape[0]} rows')
print(f'Train attrition rate: {y_train.mean()*100:.1f}%')
print(f'Test attrition rate: {y_test.mean()*100:.1f}%')

## 4. Scale Features (for Logistic Regression)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save scaler for future use
joblib.dump(scaler, '../models/scaler.pkl')
print('Scaler fitted and saved.')

## 5. Model 1 — Logistic Regression (Baseline)
Good for: Interpretability and establishing a baseline. Coefficients tell us the direction of each feature's influence.

In [ ]:
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

print('=== Logistic Regression Results ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_lr):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_lr):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred_lr):.4f}')
print(f'F1 Score:  {f1_score(y_test, y_pred_lr):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob_lr):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_lr, target_names=['Stayed', 'Left']))

## 6. Model 2 — Random Forest (Main Model)
Good for: Capturing non-linear relationships, interactions between features, and providing feature importance.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print('=== Random Forest Results ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_rf):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred_rf):.4f}')
print(f'F1 Score:  {f1_score(y_test, y_pred_rf):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob_rf):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_rf, target_names=['Stayed', 'Left']))

## 7. Model Comparison — Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, title in [
    (axes[0], y_pred_lr, 'Logistic Regression'),
    (axes[1], y_pred_rf, 'Random Forest')
]:
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Stayed', 'Left'],
                yticklabels=['Stayed', 'Left'])
    ax.set_title(f'Confusion Matrix — {title}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('../reports/fig9_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. ROC Curve Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for y_prob, label, color in [
    (y_prob_lr, f'Logistic Regression (AUC={roc_auc_score(y_test, y_prob_lr):.3f})', '#3498db'),
    (y_prob_rf, f'Random Forest (AUC={roc_auc_score(y_test, y_prob_rf):.3f})', '#e74c3c')
]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    ax.plot(fpr, tpr, label=label, color=color, linewidth=2.5)

ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier (AUC=0.500)', linewidth=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig('../reports/fig10_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Cross Validation

In [ ]:
cv_scores_lr = cross_val_score(lr, X_train_scaled, y_train, cv=5, scoring='roc_auc')
cv_scores_rf = cross_val_score(rf, X_train, y_train, cv=5, scoring='roc_auc')

print('5-Fold Cross Validation ROC-AUC:')
print(f'Logistic Regression: {cv_scores_lr.mean():.4f} ± {cv_scores_lr.std():.4f}')
print(f'Random Forest:       {cv_scores_rf.mean():.4f} ± {cv_scores_rf.std():.4f}')

## 10. Save Best Model

In [ ]:
# Save Random Forest as the primary model
joblib.dump(rf, '../models/rf_attrition_model.pkl')
joblib.dump(lr, '../models/lr_attrition_model.pkl')

# Save feature names for later use
feature_names = list(X.columns)
joblib.dump(feature_names, '../models/feature_names.pkl')

print('Models saved to /models/')
print(f'Features saved: {len(feature_names)} features')

In [ ]:
import joblib
import os

os.makedirs("models", exist_ok=True)

joblib.dump(rf, "models/rf_attrition_model.pkl")
joblib.dump(list(X_train.columns), "models/feature_names.pkl")

print("Model saved successfully!")

NameError: name 'rf' is not defined